<a href="https://colab.research.google.com/github/hanidew/G12_IR_Evaluation/blob/precision-calculation/IRWS_Precision_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2: Precision Calculation

## 2.1 Initialize the Environment and Load Data

*   Load the cleaned system runs and qrels
*   Print both datasets to ensure the data is loaded correctly





In [1]:
# import the Pandas library for handling and analyzing data
import pandas as pd

# Load the cleaned system runs ('clean_system_runs_FINAL.csv' file)
# store it in dataframe df_runs
df_runs = pd.read_csv('clean_system_runs_FINAL.csv')

# Qrels format is usually: TopicID Iteration DocID Relevance
# create a list of column names
# These names describe what each column in the Qrels file represents
qrels_cols = ['topic_id', 'iteration', 'doc_id', 'relevance']
# Load the Qrels file (qrels.trec8.adhoc' file)
# sep='\s+' : the file is not comma-separated, it is separated by spaces/tabs
# names=qrels_cols : assigns the column names
df_qrels = pd.read_csv('qrels.trec8.adhoc', sep=r'\s+', names=qrels_cols)

# Preview the data to ensure the data is loaded correctly
print("System Runs:")
print(df_runs.head(15))
print("\nQrels:")
print(df_qrels.head(15))

System Runs:
    Topic_ID Iteration       Doc_ID  Rank     Score System_Name
0        401        Q0  FT932-10861     1  0.466239   CL99XTopt
1        401        Q0  FBIS3-30642     2  0.432951   CL99XTopt
2        401        Q0  FBIS3-40017     3  0.429149   CL99XTopt
3        401        Q0  FBIS4-18182     4  0.425154   CL99XTopt
4        401        Q0  FT934-15047     5  0.424532   CL99XTopt
5        401        Q0   FBIS3-5584     6  0.423666   CL99XTopt
6        401        Q0  FBIS3-38677     7  0.423611   CL99XTopt
7        401        Q0  FBIS4-23811     8  0.423541   CL99XTopt
8        401        Q0  FT931-16878     9  0.423072   CL99XTopt
9        401        Q0   FT924-7265    10  0.422821   CL99XTopt
10       401        Q0  FT932-13416    11  0.422771   CL99XTopt
11       401        Q0  FBIS3-18684    12  0.422190   CL99XTopt
12       401        Q0   FBIS3-8571    13  0.422140   CL99XTopt
13       401        Q0  FBIS4-64863    14  0.421991   CL99XTopt
14       401        Q0  FBI

## 2.2 Define the Precision@k Function


*   Take top K retrieved documents
*   Get all relevant documents
*   Find the intersection (overlap)
*   Divide by K





In [2]:
# Define a function named calculate_precision_at_k
# the function takes 3 inputs
# system_results : the retrieved documents (ranking output)
# qrels : ground truth relevance judgments
# k : how many top results you want to evaluate
# basically, this function calculates: “Out of the top K results, how many are actually relevant?”
def calculate_precision_at_k(system_results, qrels, k):

    # Ensure correct ranking order
    # commented -> system_results = system_results.sort_values(by='Rank')

    # 1. Get Top k Documents
    # system_results.head(k) : Takes the first K rows (top-ranked documents)
    # ['Doc_ID'] : Selects only the Doc_ID column
    # .tolist() : Converts it into a list
    top_k_docs = system_results.head(k)['Doc_ID'].tolist()

    # 2. Get Relevant Documents from Qrels
    # qrels[qrels['relevance'] > 0] : keep only the relevant documents (relevance>0)
    # ['doc_id'] : extract document ID
    # .tolist() : convert to list
    # set() : convert to set for faster lookup and remove duplicates automatically
    relevant_docs = set(qrels[qrels['relevance'] > 0]['doc_id'].tolist())

    # 3. Count Relevant Retrieved Documents
    # set(top_k_docs) : convert top_k_docs to set
    # .intersection(relevant_docs) : find focuments that are retrieved (top K) AND relevant
    # len() : count the number to find the relevant documents in top K
    num_relevant_retrieved = len(set(top_k_docs).intersection(relevant_docs))

    # 4. Apply Precision@K Formula
    return num_relevant_retrieved / k


## 2.3 Generate the Score Matrices
*   Prepare data including systems, topics, and depths
*   For each topic, filter qrels
*   For each system, filter results
*   For each depth, compute the Precision@K
*   LStore everything in matrix form

In [3]:
# 1. Get all systems and topics
systems = df_runs['System_Name'].unique() # get all unique retrieval systems
topics = sorted(df_runs['Topic_ID'].unique()) # get all query topics and sort in ascending order

# 2. Define evaluation depths
# 5-very top results, 10-standard evaluation, 20-medium depth, 50-deep ranking, 100-very deep retrieval
depths = [5, 10, 20, 50, 100]

# 3. Prepare storage structure
results = {k: [] for k in depths} # each list will store rows of results for each topic

# 4. Loop trhough each topic
for topic in topics:
    # create structure for each topic
    topic_results = {k: {'Topic': topic} for k in depths}
    # get qrels for the topic
    current_qrels = df_qrels[df_qrels['topic_id'] == topic]

    # 5. Loop through each system
    for system in systems:

        # filter system results to extract:
        current_run = df_runs[
            (df_runs['System_Name'] == system) & # only results from this system
            (df_runs['Topic_ID'] == topic) # only for this topic
        ].sort_values(by='Rank')

        # 6. Loop through all depths
        for k in depths:

            # handle empty results to prevent errors and keep matrix complete
            if current_run.empty: # if a system returned nothing
                topic_results[k][system] = 0  # precision=0

            # compute Precision@K
            else:
                # call the function, take top K results, compare with relevant docs, return precision
                topic_results[k][system] = calculate_precision_at_k(current_run, current_qrels, k)

    # 7. Save results per topic
    for k in depths:
        results[k].append(topic_results[k])

# 8. Convert to matrices
matrices = {
    # convert list to dataframe, set topic as row index
    k: pd.DataFrame(results[k]).set_index('Topic')
    for k in depths # for each depth
}

# 9. Display results
for k in depths:
    print(f"Precision@{k} Matrix Preview:")
    print(matrices[k].head())
    print("\n")

# 10. mean scores
mean_scores = {}

for k in depths:
    mean_scores[k] = matrices[k].mean().sort_values(ascending=False)

    print(f"\nMean Precision@{k} per System:")
    print(mean_scores[k])

# Convert mean scores into DataFrame (optional but recommended)
mean_matrix = pd.DataFrame(mean_scores)

print(matrices[5]



Precision@5 Matrix Preview:
       CL99XTopt  Flab8at  GE8ATDN2  INQ602  Mer8Adtd4  acsys8alo2  apl8c621  \
Topic                                                                          
401          0.0      0.2       0.8     0.0        0.2         0.0       1.0   
402          0.6      0.8       1.0     0.8        0.6         0.8       1.0   
403          1.0      1.0       1.0     1.0        1.0         1.0       1.0   
404          0.6      0.0       0.6     0.4        0.4         0.4       0.6   
405          0.6      0.6       0.4     0.6        0.4         0.8       0.8   

       att99ate  fub99td  ibms99c  isa50t  mds08a2  nttd8alx  orcl99man  \
Topic                                                                     
401         0.4      0.6      1.0     0.2      1.0       0.0        0.4   
402         0.8      1.0      1.0     0.6      1.0       0.6        0.8   
403         0.8      1.0      1.0     0.8      1.0       0.8        1.0   
404         0.6      0.6      0.6   

In [5]:
print(matrices[10].loc[432])
print("\n")
print(matrices[10].loc[437])

CL99XTopt     0.1
Flab8at       0.0
GE8ATDN2      0.0
INQ602        0.2
Mer8Adtd4     0.0
acsys8alo2    0.0
apl8c621      0.0
att99ate      0.0
fub99td       0.0
ibms99c       0.0
isa50t        0.0
mds08a2       0.7
nttd8alx      0.0
orcl99man     0.0
plt8ah1       0.0
Name: 432, dtype: float64


CL99XTopt     0.0
Flab8at       0.0
GE8ATDN2      0.0
INQ602        0.0
Mer8Adtd4     0.1
acsys8alo2    0.0
apl8c621      0.0
att99ate      0.4
fub99td       0.0
ibms99c       0.2
isa50t        0.1
mds08a2       0.0
nttd8alx      0.1
orcl99man     0.0
plt8ah1       0.0
Name: 437, dtype: float64


## 2.4 Export the matrices in csv format

In [4]:
# Export the precision matroces to CSV format
for k in depths:
    matrices[k].to_csv(f'P@{k}.csv')

mean_matrix.to_csv("Mean_Precision_per_System.csv")

print("Phase 1 Complete: Matrices for Precision@5,10,20,50, and 100 generated and saved.")

Phase 1 Complete: Matrices for Precision@5,10,20,50, and 100 generated and saved.
